# 第09章 图像梯度（跟随代码）

> 《OpenCV轻松入门：面向Python》第 9 章的跟随代码，内容未做改动。
> 运行前请确认工作目录为本文件夹（picture*.jpg 等素材都在这里）。

# 第九章 图像梯度

## Sobel算子

`dst=cv2.Sobel(src,ddepth,dx,dy,ksize,scale,delta,borderType)`
- `src`：输入图像，单通道或三通道，`uint8`类型。
- `ddepth`：输出图像的深度，`cv2.CV_16S`或`cv2.CV_32F`类型。
- `dx`：x方向上的卷积核，`int`类型。
- `dy`：y方向上的卷积核，`int`类型。
- `ksize`：卷积核大小，`int`类型，必须为1、3、5、7。若设置为-1，则会使用Scharr算子
- `scale`：缩放因子，`double`类型。
- `delta`：阈值，`double`类型。
- `borderType`：边界类型，`cv2.BORDER_DEFAULT`或`cv2.BORDER_REPLICATE`类型。


$$
dst=saturate(src\times kernel\times scale+delta)
$$


为什么不能直接用 cv2.CV_8U (或 -1)？
原因：Sobel算子是在做减法。
从黑(0)到白(255)的边缘，导数是正数。
从白(255)到黑(0)的边缘，导数是负数。
如果你设定输出深度为 uint8，所有的负数会被OpenCV自动截断为0（就像你之前遇到的黑帽运算问题一样）。这意味着你会丢失所有由亮变暗的边缘！
正确做法：使用 cv2.CV_64F (64位浮点数) 或 cv2.CV_16S (16位有符号整数) 来保留负梯度，最后再取绝对值转回 uint8。


### opencv中对参数去绝对值

`dst=cv2.convertScaleAbs(src,alpha,beta)`
$$
dst=saturate(src\times alpha+beta)
$$

In [3]:
import cv2

import numpy as np

img=np.random.randint(-256,256,size=(5,5),dtype=np.int16)

rst=cv2.convertScaleAbs(img,alpha=1,beta=2)

print(img)
print(rst)

[[-200  112   -7   97 -232]
 [ -70   63  -36  -42   52]
 [  73   -7  -77  -48  172]
 [ 110    1  215  204 -145]
 [-200   41  139  -97  232]]
[[198 114   5  99 230]
 [ 68  65  34  40  54]
 [ 75   5  75  46 174]
 [112   3 217 206 143]
 [198  43 141  95 234]]


In [11]:
#使用梯度增强边缘
img=cv2.imread('picture2.png',0)
#手动锐化
Gx=np.array([[-1,0,1],[-2,0,2],[-1,0,1]])
Gy=np.array([[-1,-2,-1],[0,0,0],[1,2,1]])
img_x=cv2.filter2D(img,cv2.CV_64F,Gx)
img_y=cv2.filter2D(img,cv2.CV_64F,Gy)
img_xy1=np.sqrt(img_x**2+img_y**2)
img_xy2=np.abs(img_x)+np.abs(img_y)
img_xy=np.maximum(img_xy1,img_xy2)
img_xy=np.clip(img_xy,0,255).astype(np.uint8)
cv_show("img",img)
cv_show("img_xy",img_xy)

In [237]:
#计算x方向边缘梯度：dx=1,dy=0

img=cv2.imread("picture7.png",cv2.IMREAD_GRAYSCALE)
dst1=cv2.Sobel(img,cv2.CV_64F,dx=1,dy=0)
dst1=cv2.convertScaleAbs(dst1)
cv_show("Sobel",dst1)

#计算y方向边缘梯度：dx=0,dy=1

dst2=cv2.Sobel(img,cv2.CV_64F,dx=0,dy=1)
dst2=cv2.convertScaleAbs(dst2)
cv_show("Sobel",dst2)

#计算x方向边缘梯度和y方向边缘梯度的和：dx=1,dy=1

dst3=cv2.Sobel(img,cv2.CV_64F,dx=1,dy=1)
dst3=cv2.convertScaleAbs(dst3)
cv_show("Sobel",dst3)

dst4=cv2.addWeighted(dst1,0.5,dst2,0.5,0)
cv_show("Sobel",dst4)

In [12]:
img=cv2.imread("picture2.png",cv2.IMREAD_GRAYSCALE)

#方法1
dst1=cv2.Sobel(img,cv2.CV_16S,1,0,ksize=3)
dst2=cv2.Sobel(img,cv2.CV_16S,0,1,ksize=3)

dst1=cv2.convertScaleAbs(dst1)
dst2=cv2.convertScaleAbs(dst2)

dst3=cv2.addWeighted(dst1,0.5,dst2,0.5,0)

cv_show("dst1",dst1)
cv_show("dst2",dst2)
cv_show("dst3",dst3)

#方法2
dst=cv2.Sobel(img,cv2.CV_16S,1,1,ksize=3)

dst=cv2.convertScaleAbs(dst)

cv_show("dst",dst)

## Scharr与Sobel的区别

- 卷积核权重：Scharr核权重为3, 10, 3，Sobel核权重为1, 2, 1。
- 精度：Scharr核精度高于Sobel核。
- 敏感度：Scharr核对边缘检测敏感，Sobel核对角线检测敏感。
- 应用：Scharr高精度梯度，Sobel通用边缘检测，不需要极高精度。
- 尺寸限制：Scharr核尺寸只能为3x3，Sobel核尺寸为1、3、5、7。

## Scharr算子

`dst=cv2.Scharr(src,ddepth,dx,dy,scale,delta,borderType)`
- `src`：输入图像，单通道或三通道。
- `ddepth`：输出图像的深度，默认为-1，表示与输入图像相同。
- `dx`：x方向上的求导阶数，可以为1、2、3。
- `dy`：y方向上的求导阶数，可以为0、1、2。
- `scale`：缩放因子，默认为1。
- `delta`：阈值，默认为0。
- `borderType`：边界填充方式，默认为cv2.BORDER_DEFAULT。



In [6]:
#计算x方向边缘梯度：dx=1,dy=0

img=cv2.imread("picture2.png",cv2.IMREAD_GRAYSCALE)
dst1=cv2.Scharr(img,cv2.CV_64F,dx=1,dy=0)
dst1=cv2.convertScaleAbs(dst1)
cv_show("Sobel",dst1)

#计算y方向边缘梯度：dx=0,dy=1

dst2=cv2.Scharr(img,cv2.CV_64F,dx=0,dy=1)
dst2=cv2.convertScaleAbs(dst2)
cv_show("Sobel",dst2)

dst4=cv2.addWeighted(dst1,0.5,dst2,0.5,0)
cv_show("Sobel",dst4)

In [7]:
img=cv2.imread("picture1.jpg",cv2.IMREAD_GRAYSCALE)

#方法1
dst1=cv2.Scharr(img,cv2.CV_16S,1,0,scale=1)
dst2=cv2.Scharr(img,cv2.CV_16S,0,1,scale=1)

dst1=cv2.convertScaleAbs(dst1)
dst2=cv2.convertScaleAbs(dst2)

dst3=cv2.addWeighted(dst1,0.5,dst2,0.5,0)

cv_show("img",img)
cv_show("dst1",dst1)
cv_show("dst2",dst2)
cv_show("dst3",dst3)

## Laplacian算子

In [8]:
img=cv2.imread("picture7.png",cv2.IMREAD_GRAYSCALE)

laplacian=cv2.Laplacian(img,cv2.CV_64F,3)
laplacian=cv2.convertScaleAbs(laplacian)
cv_show("Laplacian",laplacian)

In [9]:
img=cv2.imread("picture2.png",cv2.IMREAD_GRAYSCALE)

laplacian=cv2.Laplacian(img,cv2.CV_64F)
laplacian=cv2.convertScaleAbs(laplacian)

cv_show("Laplacian",laplacian)